[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-01-actions-fundamentals.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · GitHub Actions Fundamentals — Workflows, Jobs, Steps, and Triggers
**certified-journeys / github-actions-certified** · Day 1 · Fundamentals

> **Goal for today:** By the end of this notebook you can write a multi-job workflow from scratch, wire up `needs:` dependencies, and reason about which trigger fires when.


In [ ]:
%pip install -q pyyaml


## Step 1 · Anatomy of a GitHub Actions workflow file

A workflow file is a YAML document stored under `.github/workflows/`. Every file must declare:

| Key | Required | What it does |
|-----|----------|--------------|
| `name` | no | Display name in the Actions UI |
| `on` | **yes** | Events that trigger the workflow |
| `jobs` | **yes** | One or more jobs to run |

Each **job** contains:
- `runs-on` — the type of runner (`ubuntu-latest`, `windows-latest`, etc.)
- `steps` — ordered list of shell commands (`run:`) or Actions (`uses:`)

Jobs run **in parallel by default**. Use `needs:` to serialize them.


In [ ]:
import pathlib, yaml, json, textwrap

# Write our first workflow to .github/workflows/ so we can inspect it
wf_dir = pathlib.Path(".github/workflows")
wf_dir.mkdir(parents=True, exist_ok=True)

hello_world_yaml = """\
name: Hello World

on:
  push:
    branches: [main]

jobs:
  greet:
    runs-on: ubuntu-latest
    steps:
      - name: Say hello
        run: echo "Hello, GitHub Actions!"
"""

wf_path = wf_dir / "hello-world.yml"
wf_path.write_text(hello_world_yaml)
print(f"Written to {wf_path}")

# Parse and pretty-print the structure so we can explore it programmatically
parsed = yaml.safe_load(hello_world_yaml)
print(json.dumps(parsed, indent=2))


### What just happened?

- We wrote a **valid workflow file** to `.github/workflows/hello-world.yml` — exactly where GitHub Actions looks.
- **`on: push: branches: [main]`** means this workflow fires whenever a commit is pushed to `main`.
- The single job `greet` runs on an Ubuntu runner and executes one `run:` step.
- **`yaml.safe_load`** parses the file into a Python dict — useful for programmatic validation (we'll do this more in Day 2).


## Step 2 · Adding a second job with `needs:`

Without `needs:`, **all jobs start simultaneously**. This causes the classic deploy-before-test race condition.

```
  build ──┐
           ├─ test (needs: build)
  lint  ──┘           └─ deploy (needs: test)
```

`needs:` accepts a single job name **or** a list. A job only starts after all its dependencies succeed.

Key rules:
- `needs:` creates a **DAG** (directed acyclic graph), not a linear sequence.
- If any dependency fails, the dependent job is skipped by default.
- Use `if: always()` to override that behaviour.


In [ ]:
two_job_yaml = """\
name: Build and Test

on:
  push:
    branches: [main]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - name: Compile
        run: echo "Building the project..."

  test:
    runs-on: ubuntu-latest
    needs: build          # <-- test waits for build to succeed
    steps:
      - name: Run tests
        run: echo "Running test suite..."
"""

wf_path2 = wf_dir / "build-and-test.yml"
wf_path2.write_text(two_job_yaml)

# Visualise the dependency graph in text form
parsed2 = yaml.safe_load(two_job_yaml)
jobs = parsed2["jobs"]

print("Job dependency graph:")
for name, cfg in jobs.items():
    deps = cfg.get("needs", [])
    if isinstance(deps, str):
        deps = [deps]    # normalise single string to list
    if deps:
        print(f"  {name}  ←  needs: {', '.join(deps)}")
    else:
        print(f"  {name}  (no dependencies — starts immediately)")


### What just happened?

- We added a second job `test` and gave it **`needs: build`**.
- The Python snippet extracts the dependency graph from the parsed YAML — a useful technique for linting workflows programmatically.
- **`needs` accepts a string or a list** — our normalisation `if isinstance(deps, str): deps = [deps]` handles both forms safely.
- Without this serialisation, `test` would launch at the same time as `build`, potentially testing un-built artefacts.


## Step 3 · Trigger events — `push`, `pull_request`, and `workflow_dispatch`

The `on:` key controls **when** a workflow runs. The three most common triggers:

| Trigger | Fires when | Common use |
|---------|-----------|------------|
| `push` | A commit is pushed to a matching branch/tag | CI on every commit |
| `pull_request` | A PR is opened, synchronised, or re-opened | Pre-merge validation |
| `workflow_dispatch` | You click "Run workflow" in the UI (or call the API) | Manual / on-demand runs |

You can combine all three under a single `on:` block. GitHub will fire the workflow for whichever event actually occurs.

**Important gotcha:** `pull_request` from a fork runs with read-only permissions by default (no secrets). Use `pull_request_target` with extreme caution — it runs in the base repo context and has access to secrets, so it's a security risk if you check out PR code.


In [ ]:
multi_trigger_yaml = """\
name: CI — Multi-Trigger

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]
  workflow_dispatch:         # enables manual trigger from the Actions UI
    inputs:
      debug_enabled:
        description: 'Run with tmate debugging?'
        required: false
        default: 'false'

jobs:
  ci:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Show trigger context
        run: |
          echo "Event name : ${{ github.event_name }}"
          echo "Ref        : ${{ github.ref }}"
          echo "SHA        : ${{ github.sha }}"
"""

wf_path3 = wf_dir / "ci-multi-trigger.yml"
wf_path3.write_text(multi_trigger_yaml)

# Show which branches each trigger watches
parsed3 = yaml.safe_load(multi_trigger_yaml)
on_block = parsed3["on"]

print("Registered triggers:")
for event, config in on_block.items():
    if config and "branches" in config:
        print(f"  {event}: watches branches → {config['branches']}")
    elif config and "inputs" in config:
        inputs = list(config["inputs"].keys())
        print(f"  {event}: manual — inputs: {inputs}")
    else:
        print(f"  {event}: (no branch filter)")


### What just happened?

- We registered **three triggers** in one `on:` block — GitHub will fire this workflow on any matching event.
- **`workflow_dispatch` with `inputs:`** adds a form in the GitHub UI where you can pass parameters before running. Great for controlled deployments.
- The `run:` step uses **expression syntax** (`${{ github.event_name }}`) to print the triggering event at runtime — helpful for debugging which path fired.
- **`actions/checkout@v4`** is always the first step in almost every real workflow — without it, the runner's workspace is empty.


## Step 4 · Reading the events reference — key event categories

The [events-that-trigger-workflows](https://docs.github.com/en/actions/writing-workflows/choosing-when-your-workflow-runs/events-that-trigger-workflows) reference lists 35+ events. For MLOps the most relevant are:

| Category | Events | Use |
|----------|--------|-----|
| Code changes | `push`, `pull_request` | Standard CI |
| Scheduling | `schedule` (cron) | Nightly retraining, drift checks |
| Manual | `workflow_dispatch` | On-demand deploys, one-off jobs |
| Cross-workflow | `workflow_call` | Reusable workflow entry point (Day 3) |
| Repository | `release`, `create` | Release pipelines, tag-triggered builds |
| Webhooks | `repository_dispatch` | External triggers (e.g. new dataset landed in S3) |

Each event can have **activity types** — for example, `pull_request` fires on `opened`, `synchronize`, `reopened`, `closed`. You can filter to specific types:

```yaml
on:
  pull_request:
    types: [opened, synchronize]
```


In [ ]:
# Demonstrate a schedule trigger — useful for nightly model retraining
schedule_yaml = """\
name: Nightly Model Retrain

on:
  schedule:
    - cron: '0 2 * * *'    # 02:00 UTC every day
  workflow_dispatch: {}     # also allow manual runs

jobs:
  retrain:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Train model
        run: python train.py --config configs/nightly.yaml
      - name: Evaluate and push metrics
        run: python evaluate.py --push-to-mlflow
"""

(wf_dir / "nightly-retrain.yml").write_text(schedule_yaml)

# Parse the cron expression and explain it
import re

parsed_sched = yaml.safe_load(schedule_yaml)
cron = parsed_sched["on"]["schedule"][0]["cron"]

fields = cron.split()
labels = ["minute", "hour", "day-of-month", "month", "day-of-week"]
print(f"Cron expression: {cron}")
print("Breakdown:")
for label, val in zip(labels, fields):
    desc = "*" if val == "*" else val
    print(f"  {label:15s} = {desc}")
print("\nFires: every day at 02:00 UTC")


### What just happened?

- **`schedule` with `cron:`** fires the workflow on a time-based schedule — perfect for nightly retraining pipelines.
- GitHub Actions uses UTC for all cron schedules; plan accordingly for team timezones.
- We also added `workflow_dispatch: {}` — the empty dict is valid YAML and lets you trigger the same workflow manually when you need an out-of-schedule retrain.
- **Warning:** GitHub may delay scheduled workflows by up to 15 minutes during peak load. Don't rely on exact timing for time-sensitive operations.


## Step 5 · Validating workflow YAML locally

Before pushing, you can catch many errors locally:

1. **PyYAML parse check** — catches syntax errors (wrong indentation, missing quotes).
2. **Schema validation** — `actionlint` (a Go binary) enforces the Actions schema. On Colab we approximate this with structural checks in Python.
3. **Required field check** — every workflow must have `on:` and `jobs:`.

Below we build a minimal workflow validator that catches the most common mistakes.


In [ ]:
def validate_workflow(yaml_text: str, filename: str = "workflow.yml") -> list[str]:
    """Return a list of error strings. Empty list = valid."""
    errors = []

    # 1. YAML parse
    try:
        wf = yaml.safe_load(yaml_text)
    except yaml.YAMLError as exc:
        return [f"YAML parse error: {exc}"]

    if not isinstance(wf, dict):
        return ["Workflow must be a YAML mapping at the top level"]

    # 2. Required top-level keys
    for key in ("on", "jobs"):
        if key not in wf:
            errors.append(f"Missing required top-level key: '{key}'")

    if "jobs" not in wf:
        return errors   # nothing more to check without jobs

    # 3. Each job must have runs-on and steps
    for job_name, job_cfg in wf["jobs"].items():
        if not isinstance(job_cfg, dict):
            errors.append(f"Job '{job_name}' must be a mapping")
            continue
        if "runs-on" not in job_cfg:
            errors.append(f"Job '{job_name}': missing 'runs-on'")
        if "steps" not in job_cfg:
            errors.append(f"Job '{job_name}': missing 'steps'")

    # 4. Detect needs: references to unknown jobs
    known_jobs = set(wf["jobs"].keys())
    for job_name, job_cfg in wf["jobs"].items():
        if not isinstance(job_cfg, dict):
            continue
        deps = job_cfg.get("needs", [])
        if isinstance(deps, str):
            deps = [deps]
        for dep in deps:
            if dep not in known_jobs:
                errors.append(f"Job '{job_name}': needs: '{dep}' — job not defined")

    return errors


# --- Test the validator ---

# Valid workflow
errs = validate_workflow(two_job_yaml, "build-and-test.yml")
print("build-and-test.yml:", "✓ valid" if not errs else errs)

# Broken workflow — missing runs-on, dangling needs:
broken = """\
name: Broken
on: push
jobs:
  deploy:
    needs: build          # 'build' job doesn't exist!
    steps:
      - run: echo hi
"""
errs2 = validate_workflow(broken, "broken.yml")
print("\nbroken.yml errors:")
for e in errs2:
    print(" •", e)


### What just happened?

- We built a **four-check validator**: parse, required keys, per-job required fields, and dangling `needs:` references.
- The broken workflow catches two errors: missing `runs-on` and a `needs:` that points to a non-existent job.
- **This is the pattern `actionlint` uses** — parse the YAML graph, then walk it checking invariants. Install the real `actionlint` binary for comprehensive checks (including expression syntax).
- For CI, pipe this check through `actionlint` in a pre-commit hook or a separate lint job so bad workflows are rejected before they hit your main branch.


In [ ]:
# Challenge: Build a workflow that:
#   1. Triggers on push to main AND on workflow_dispatch
#   2. Has three jobs: lint → test → deploy (serialised with needs:)
#   3. Each job echoes its name and the triggering event
#   4. Passes your validate_workflow() with zero errors
#
# Scaffold — fill in the YAML string and run the validator:

challenge_yaml = """\
name: Three-Stage Pipeline

on:
  # TODO: add push (branches: [main]) and workflow_dispatch

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      # TODO: echo job name and ${{ github.event_name }}

  test:
    runs-on: ubuntu-latest
    # TODO: add needs: lint
    steps:
      # TODO: echo job name

  deploy:
    runs-on: ubuntu-latest
    # TODO: add needs: test
    steps:
      # TODO: echo job name
"""

# Validate your solution
errors = validate_workflow(challenge_yaml)
if errors:
    print("Validation errors:")
    for e in errors:
        print(" •", e)
else:
    print("✓ Workflow is valid!")
    pathlib.Path(".github/workflows/three-stage.yml").write_text(challenge_yaml)
    print("Written to .github/workflows/three-stage.yml")


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Workflow file location | `.github/workflows/*.yml` — GitHub scans this directory automatically |
| `on:` key | Controls which events fire the workflow; combine multiple triggers |
| Jobs | Run in parallel by default; `needs:` is the only serialisation mechanism |
| `needs:` | Accepts a string or list; creates a DAG; failed deps skip dependents |
| `runs-on` | Selects the runner OS; `ubuntu-latest` is the most common |
| `steps` | Ordered list within a job; each step is either `run:` or `uses:` |
| `workflow_dispatch` | Enables manual runs with optional typed inputs |
| `schedule` | Cron-based triggers in UTC; up to ~15 min jitter under load |

> **Tip:** Jobs run in parallel by default. `needs:` is the only thing that serializes them. If you forget `needs:`, your deploy job will race your test job — and sometimes win.

---
## What's next
**Day 2** → Deep dive into YAML syntax: environment variables at workflow/job/step scope, multi-line `run:` scripts, and the `contexts` reference.

Mark Day 1 complete in your [tracker](../index.html).
